# Grounding a Model in Your Own Corpus (RAG)

**Computational Text Analysis Pack · Module 27 — Generative Models in DH Research**

*Technique:* Retrieval-augmented generation over course corpora; citation you can verify

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/14-generative-llms/27b_rag_grounding.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

The previous notebook's failures all share one cause: the model was answering
from memory. **Retrieval-augmented generation** removes that. Retrieve passages
from a corpus *you* control, put them in the prompt, and require the answer to
come from them.

This is the only generative architecture that is straightforwardly defensible
in scholarship, because every claim traces to a passage you can open and read —
the same standard the concordance set in module 21.

## The tooling

**HuggingFace `transformers`** runs open models locally, including the decoding
strategies below. **vLLM** and **llama.cpp** are the usual serving layers when
you need throughput. Hosted APIs — OpenAI, Anthropic, Cohere — are what most
researchers actually reach for.

In CourseHub these cells go through a **gateway** that holds the credential
server-side and routes to open models. In Colab you supply your own key, or the
notebook falls back to recorded output. No API key appears in any cell of this
course, because a notebook cell is readable by whoever runs it.

**The cost question is a methods question.** Generating over a 5,000-document
corpus is not free and not fast, and unlike a frequency count it is not
deterministic — two runs give two answers. Budget and reproducibility both bite.

In [ ]:
import sys, os, json, textwrap

IN_COLAB = "google.colab" in sys.modules
LLM_BACKEND = "recorded"
_chat = None

if not IN_COLAB:
    try:
        sys.path.insert(0, "/app")
        from utils.llm_gateway import gateway_chat as _gw
        def _chat(messages, model="gpt-oss-20b", max_tokens=300):
            # allow_fallback=False keeps us on the local provider; the fallback
            # is billed per token and a classroom can run a lot of tokens.
            txt, meta = _gw(messages, model=model, max_tokens=max_tokens,
                            allow_fallback=False)
            return txt, meta.get("served_model", model)
        _chat([{"role": "user", "content": "ok"}], max_tokens=5)
        LLM_BACKEND = "coursehub-gateway"
    except Exception as exc:
        _why = str(exc)[:80]
else:
    key = os.environ.get("OPENAI_API_KEY", "")
    if key:
        def _chat(messages, model="gpt-4o-mini", max_tokens=300):
            import requests
            r = requests.post("https://api.openai.com/v1/chat/completions",
                              headers={"Authorization": f"Bearer {key}"},
                              json={"model": model, "messages": messages,
                                    "max_tokens": max_tokens}, timeout=60)
            r.raise_for_status()
            return r.json()["choices"][0]["message"]["content"], model
        LLM_BACKEND = "colab-openai"
    else:
        _why = "no OPENAI_API_KEY set in this Colab session"

def ask(prompt, model=None, max_tokens=300, system=None):
    """One prompt in, one string out — or a recorded answer when offline."""
    if _chat is None:
        return None, "recorded"
    msgs = ([{"role": "system", "content": system}] if system else []) + \
           [{"role": "user", "content": prompt}]
    kw = {"max_tokens": max_tokens}
    if model:
        kw["model"] = model
    return _chat(msgs, **kw)

print(f"LLM backend: {LLM_BACKEND}")
if LLM_BACKEND == "recorded":
    print(f"   ({_why})")
    print("   Cells below print outputs recorded when this notebook was written,")
    print("   clearly marked, so the lesson still works without a model.")
elif LLM_BACKEND == "colab-openai":
    print("   using your own key from this Colab session")
else:
    print("   using the CourseHub gateway; no key needed and none is exposed here")

In [ ]:
import numpy as np

# Retrieval uses the LaBSE vectors shipped with this course (module 23), so this
# works with no GPU and no pod.
bank = np.load("labse_headlines.npy")
index = pd.read_csv("labse_headlines_index.csv")
print(f"retrieval index: {bank.shape[0]} passages, {bank.shape[1]} dims, "
      f"{index['language'].nunique()} languages")

def retrieve(query_vec, k=5):
    scores = bank @ query_vec
    hits = np.argsort(scores)[::-1][:k]
    return [(float(scores[i]), index.iloc[i]) for i in hits]

## Retrieval without a GPU

We need a vector for the query. If module 23's backend is available we embed it
properly; otherwise we fall back to lexical overlap, which is worse but honest
and keeps the lesson runnable.

In [ ]:
def query_vector(text):
    try:
        sys.path.insert(0, "/app")
        from utils import gpu_pod
        return np.asarray(gpu_pod.embed([text])["vectors"][0], dtype="float32")
    except Exception:
        return None

QUESTION = "What have governments announced about public health?"   # <- change

qv = query_vector(QUESTION)
if qv is not None:
    hits = retrieve(qv, k=5)
    mode = "LaBSE semantic retrieval"
else:
    # Lexical fallback: crude, but it retrieves something checkable.
    terms = set(str(QUESTION).lower().split())
    scored = []
    for i, r in index.iterrows():
        overlap = len(terms & set(str(r["headline"]).lower().split()))
        if overlap:
            scored.append((overlap, r))
    scored.sort(key=lambda t: -t[0])
    hits = [(float(s), r) for s, r in scored[:5]]
    mode = "lexical overlap (no embedding backend)"

print(f"retrieval mode: {mode}")
print(f"question: {QUESTION}\n")
for score, r in hits:
    print(f"  [{score:.3f}] ({r['language']}) {str(r['headline'])[:78]}")

## Generation, constrained to what was retrieved

The prompt does two things that matter: it supplies the passages, and it
instructs the model to refuse when they do not contain the answer. That refusal
instruction is the whole safeguard — without it the model falls back on memory
and you are back in notebook 27a.

In [ ]:
context = "\n".join(
    f"[{i+1}] ({r['language']}) {r['headline']}" for i, (_, r) in enumerate(hits))

SYSTEM = ("You are a careful research assistant. Answer ONLY from the numbered "
          "passages provided. Cite the passage numbers you used in square "
          "brackets. If the passages do not contain the answer, say exactly: "
          "'The provided passages do not answer this question.' Never use "
          "outside knowledge.")
PROMPT = f"Passages:\n{context}\n\nQuestion: {QUESTION}\nAnswer:"

if LLM_BACKEND != "recorded":
    out, served = ask(PROMPT, system=SYSTEM, max_tokens=220)
    print(f"model: {served}\n")
    print(" ".join(out.split()))
else:
    print("RECORDED: a grounded answer citing [1] and [3], or the refusal")
    print("sentence when the retrieved passages did not cover the question.")
print()
print("Every bracketed number points at a passage printed above. That is the")
print("difference between a generated claim and a citable one.")

## The test that proves the grounding works

Ask something the corpus cannot answer. A properly grounded system refuses; an
ungrounded one invents. Run this before trusting any RAG pipeline — including
your own.

In [ ]:
IMPOSSIBLE = "What did Wole Soyinka say about the 1986 Nobel Prize ceremony?"
context_same = context     # deliberately the WRONG passages for this question

if LLM_BACKEND != "recorded":
    out, served = ask(f"Passages:\n{context_same}\n\nQuestion: {IMPOSSIBLE}\nAnswer:",
                      system=SYSTEM, max_tokens=150)
    answer_text = " ".join(out.split())
    print("question:", IMPOSSIBLE)
    print("answer  :", answer_text[:300])
    refused = "do not answer" in answer_text.lower()
    print()
    print("REFUSED correctly" if refused else
          "DID NOT REFUSE — this pipeline would fabricate. Tighten the system prompt.")
else:
    print("RECORDED: the model returned the refusal sentence, which is the")
    print("behaviour you want. Models differ; test yours rather than assuming.")

### 🔧 Your turn 1

Build a RAG pipeline over a corpus of your own and run the impossible-question test on it. Does it refuse?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Smaller and older models frequently do not refuse — they blend the retrieved
passages with remembered material and produce something that looks cited but is
not. The refusal test is therefore not optional, and it should be re-run
whenever you change model or prompt. Note also that RAG bounds hallucination but
does not eliminate it: the model can still misread a passage it was given, so
the citations exist to be checked, not to be trusted.

</details>